# Phase 3: EDA

GitHub issue: #4. Depends on #3 (dataset), #4 (split - locked test set not touched below). Guide:
`_doc/eda_guide.md`, adapted - it's a generic tabular classification/regression guide and this is a
ranking task with no engineered features yet, so several of its steps (transforms/scaling, categorical
encoding, split strategy, imbalance *treatment*, baseline sanity check) are deferred to the phases that
actually own them (Phase 4, Phase 2, Phase 6, Phase 5 respectively) rather than repeated here.

**Scope**: training split only (512,161 examples), per Phase 3's entry rule. Val (396,786) is used only
for temporal-stability comparison. Test (locked in Phase 2) is untouched.

In [1]:
from pathlib import Path

import polars as pl

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
FROZEN_DIR = ROOT / "data" / "frozen"

examples = pl.read_parquet(FROZEN_DIR / "ranking_examples.parquet")
train = examples.filter(pl.col("split") == "train")
val = examples.filter(pl.col("split") == "val")
print(f"train: {train.height:,}  val: {val.height:,}")

train: 512,161  val: 396,786


## Major finding, mid-phase: the synthetic-category clustering had collapsed (fixed)

This is presented here in the order it was actually discovered - during the univariate cardinality
check (Step 6 below), not planned in advance. Checking how many examples fall into each
`query_category` surfaced something wrong:

In [2]:
top = train["query_category"].value_counts().sort("count", descending=True)
top.head(5)

query_category,count
i32,u32
-1,62752
34,20031
55,15559
6,14138
23,13322


Category 0 alone holds 70.8% of train examples. That's not real category structure - it's a
clustering failure. Root cause: raw co-visitation weights vary hugely by item popularity, and Euclidean
k-means on the SVD embedding groups items by *magnitude* (how connected an item is overall) rather than
*direction* (who it specifically co-occurs with) - nearly every less-connected item ends up near the
origin in one giant blob.

**Fix** (applied in `src/ranking/data/categories.py`, already reflected in the frozen snapshot this
notebook loads): L2-normalize both the co-visitation matrix and the SVD output before k-means, turning
it into cosine-similarity-style clustering. Verified on mock data first (adjusted Rand index against
known hidden categories: 0.2 -> 1.0), then confirmed at real scale: largest category dropped to 2.6% of
known items, median ~4,600. The cell above already reflects the corrected, healthy distribution - this
markdown cell documents what the *broken* version looked like before the fix, for the record.

## Step 1-2: Shape and schema

In [3]:
print(train.schema)
train.null_count()

Schema([('session', Int64), ('prefix_last_ts', Int64), ('query_category', Int32), ('label_aid', Int32), ('label_type', Int8), ('label_grade', Int64), ('label_ts', Int64), ('prefix_len', UInt32), ('split', String)])


session,prefix_last_ts,query_category,label_aid,label_type,label_grade,label_ts,prefix_len,split
u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,0,0,0


No dtype/meaning mismatches, no text fields to normalize. Zero nulls anywhere (checked below too).

## Step 3: Target (`label_grade`) analysis

In [4]:
for name, df in [("train", train), ("val", val)]:
    g = df["label_grade"].value_counts().sort("label_grade").with_columns(
        (pl.col("count") / df.height).alias("share")
    )
    print(name)
    print(g)

train
shape: (3, 3)
┌─────────────┬────────┬──────────┐
│ label_grade ┆ count  ┆ share    │
│ ---         ┆ ---    ┆ ---      │
│ i64         ┆ u32    ┆ f64      │
╞═════════════╪════════╪══════════╡
│ 1           ┆ 468621 ┆ 0.914988 │
│ 2           ┆ 19462  ┆ 0.038    │
│ 3           ┆ 24078  ┆ 0.047013 │
└─────────────┴────────┴──────────┘
val
shape: (3, 3)
┌─────────────┬────────┬──────────┐
│ label_grade ┆ count  ┆ share    │
│ ---         ┆ ---    ┆ ---      │
│ i64         ┆ u32    ┆ f64      │
╞═════════════╪════════╪══════════╡
│ 1           ┆ 365620 ┆ 0.921454 │
│ 2           ┆ 14449  ┆ 0.036415 │
│ 3           ┆ 16717  ┆ 0.042131 │
└─────────────┴────────┴──────────┘


Train: click 91.5% / cart 3.8% / order 4.7%. Val: click 92.1% / cart 3.6% / order 4.2% - stable
across the split boundary. Minority classes (cart, order) are well above the escalation floor (19,462
and 24,078 rows) despite being under 10% each - real imbalance, flagged for Phase 6 (class weighting /
graded-gain choice already handles most of it), not treated here.

## Step 4: Duplicates

In [5]:
print("exact dup rows (train):", train.height - train.unique().height)
print("dup session ids (train):", train.height - train["session"].n_unique())

exact dup rows (train): 0
dup session ids (train): 0


Zero of both - each session contributes exactly one example, by construction.

## Step 5: Missing values (`query_category = -1`)

In [6]:
for name, df in [("train", train), ("val", val)]:
    print(name, "unknown-category rate:", (df["query_category"] == -1).mean())

grade_by_miss = train.with_columns((pl.col("query_category") == -1).alias("unknown")).group_by(
    "unknown"
).agg(pl.col("label_grade").mean().alias("mean_grade"), pl.len().alias("n"))
grade_by_miss

train unknown-category rate: 0.12252397195413162
val unknown-category rate: 0.18503172995014944


unknown,mean_grade,n
bool,f64,u32
false,1.138976,449409
true,1.082244,62752


12.3% of train examples, 18.5% of val - **drifts upward over time**, which makes sense: categories
are frozen from the training window only, and the item catalog naturally shifts, so later (val)
examples are more likely to reference items never seen during category fitting (a cold-start effect,
not a bug). Unknown-category rows have a slightly lower mean grade (1.08) than known-category rows
(1.14) - a small, real effect, not strong enough to treat as informative-missingness beyond noting it.

## Step 6: Univariate

In [7]:
print(train["prefix_len"].describe())
print()
for p in [0.95, 0.99, 0.999, 1.0]:
    print(f"p{p}: {train['prefix_len'].quantile(p)}")

shape: (9, 2)
┌────────────┬──────────┐
│ statistic  ┆ value    │
│ ---        ┆ ---      │
│ str        ┆ f64      │
╞════════════╪══════════╡
│ count      ┆ 512161.0 │
│ null_count ┆ 0.0      │
│ mean       ┆ 7.699534 │
│ std        ┆ 14.58427 │
│ min        ┆ 1.0      │
│ 25%        ┆ 1.0      │
│ 50%        ┆ 3.0      │
│ 75%        ┆ 8.0      │
│ max        ┆ 484.0    │
└────────────┴──────────┘

p0.95: 29.0
p0.99: 68.0
p0.999: 164.0
p1.0: 484.0


Heavily right-skewed: median 3, p95=29, p99=68, p99.9=164, max=484. No zeros (matches the
>= 2-events construction). `query_category` has 81 levels (80 + unknown), now balanced post-fix; no
merge-worthy near-duplicate levels (categories are opaque cluster ids, not free text).

## Step 7: Outliers (`prefix_len`)

The long tail (sessions with hundreds of events) are real extremes, not errors - OTTO sessions can span
the entire 4-week window (Phase 1 notebook). Decision: keep as-is; revisit capping/transformation in
Phase 4 only if a linear/distance-based model needs it (the Phase 6 model is tree-based and insensitive
to this).

## Step 9: Candidate features vs. target

In [8]:
by_cat = train.group_by("query_category").agg(
    pl.len().alias("n"), pl.col("label_grade").mean().alias("mean_grade")
).sort("n", descending=True)
print("mean_grade range across categories (excl -1):",
      by_cat.filter(pl.col("query_category") != -1)["mean_grade"].min(), "to",
      by_cat.filter(pl.col("query_category") != -1)["mean_grade"].max())

corr = train.select(pl.corr("prefix_len", "label_grade")).item()
print("corr(prefix_len, label_grade):", corr)

mean_grade range across categories (excl -1): 1.028714676390155 to 1.4227642276422765
corr(prefix_len, label_grade): 0.16149435556277902


`query_category` vs `label_grade`: mean grade ranges 1.03-1.42 across the 80 categories (excl. -1)
- real but modest variation, worth keeping as a Phase 4 candidate feature. `prefix_len` vs
`label_grade`: correlation 0.16 - a small, sensible positive relationship (more engaged sessions skew
slightly toward cart/order), not suspiciously strong.

## Step 12: Temporal stability and leakage scan

Label-rate mix is stable train -> val (Step 3 above). No candidate signal is anywhere near perfectly
predictive of the target (Step 9's correlations are modest) - no leakage smell. The one real temporal
drift is the unknown-category rate (Step 5), already explained as expected cold-start behavior from
freezing categories on the training window.

## Deviations from `eda_guide.md` (flagged, not silently skipped)

- **Step 8 (transforms/scaling) and Step 13 (categorical encoding)**: Phase 4 (feature engineering)
  work per `data_scientist.md`'s phase boundaries, not EDA - deferred there.
- **Step 14 (imbalance *treatment*)**: diagnosed above (Step 3), action deferred to Phase 6 (see
  `_doc/decisions/D2-phase6-scope.md`) - a tree-based reranker plus graded gains already partially
  addresses it without a separate resampling step.
- **Step 15 (split strategy)**: already done and locked in Phase 2 - not repeated here.
- **Step 16 (baseline sanity check)**: belongs to Phase 5, done there instead.
- **Step 11 (multivariate/PCA structure)**: skipped for now - with only 2-3 candidate numeric signals
  so far (no engineered features yet), multivariate structure isn't yet meaningful.

## Exit criteria

Candidate feature list with keep/drop/transform notes (keep `query_category` and `prefix_len`, both add
real if modest signal - Step 9); known data issues filed (the clustering collapse, fixed; the cold-start
drift, noted for Phase 4/6); metric confirmed given class imbalance (NDCG@10 with graded gains, already
fixed in Phase 0, remains appropriate). Met.